# 03 — Model Training & Evaluation

Train and compare three model architectures for pIC50 prediction:
1. **Random Forest** (classical ML baseline)
2. **XGBoost** (gradient boosting)
3. **Graph Neural Network** (molecular graph-based)

All experiments tracked with MLflow.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import torch
import yaml
import mlflow
from sklearn.model_selection import train_test_split

from src.components.model_trainer import IC50ModelTrainer
from src.components.model_evaluator import ModelEvaluator
from src.components.feature_engineering import MolecularFeatureEngineer
from src.components.gnn_model import EGFRGraphNet

%matplotlib inline

In [ ]:
with open("../configs/config.yaml") as f:
    config = yaml.safe_load(f)

curated_df = pd.read_csv("../data/processed/egfr_curated.csv")
feature_df = pd.read_csv("../data/processed/egfr_features.csv")
graph_dataset = torch.load("../data/processed/egfr_graphs.pt", weights_only=False)

print(f"Compounds: {len(curated_df)}, Features: {feature_df.shape[1]}, Graphs: {len(graph_dataset)}")

## 1. Train/Test Split

In [ ]:
X = feature_df.values
y = curated_df["pIC50"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config["model"]["test_size"],
    random_state=config["model"]["random_state"]
)

# Split graph dataset with same indices
indices = np.arange(len(graph_dataset))
train_idx, test_idx = train_test_split(
    indices, test_size=config["model"]["test_size"],
    random_state=config["model"]["random_state"]
)
train_graphs = [graph_dataset[i] for i in train_idx]
test_graphs = [graph_dataset[i] for i in test_idx]

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train graphs: {len(train_graphs)}, Test graphs: {len(test_graphs)}")

## 2. Train Models

In [ ]:
trainer = IC50ModelTrainer(config)
evaluator = ModelEvaluator(output_dir="../results")

In [ ]:
# Random Forest
rf_model = trainer.train_random_forest(X_train, y_train)
rf_pred = rf_model.predict(X_test)

In [ ]:
# XGBoost
xgb_model = trainer.train_xgboost(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

In [ ]:
# GNN
num_node_features = train_graphs[0].x.shape[1]
gnn_model = trainer.train_gnn(train_graphs, test_graphs, num_node_features)
device = trainer.device
gnn_pred = evaluator.predict_gnn(gnn_model, test_graphs, device)
gnn_y_true = np.array([g.y.item() for g in test_graphs])

## 3. Model Comparison

In [ ]:
results = {
    "RandomForest": {"y_true": y_test, "y_pred": rf_pred},
    "XGBoost": {"y_true": y_test, "y_pred": xgb_pred},
    "GNN": {"y_true": gnn_y_true, "y_pred": gnn_pred},
}

comparison = evaluator.compare_models(results)
comparison

## 4. Evaluation Plots

In [ ]:
evaluator.generate_report(results)

In [ ]:
# Feature importance for tree models
feature_names = list(pd.read_csv("../data/processed/egfr_features.csv", nrows=0).columns)
evaluator.plot_feature_importance(rf_model, feature_names, "RandomForest")
evaluator.plot_feature_importance(xgb_model, feature_names, "XGBoost")

In [ ]:
# Learning curves
evaluator.plot_learning_curves(rf_model, X_train, y_train, "RandomForest")
evaluator.plot_learning_curves(xgb_model, X_train, y_train, "XGBoost")

## 5. MLflow Experiment Summary

View all runs with: `mlflow ui` from the project root directory.

In [ ]:
# Query MLflow for logged runs
experiment = mlflow.get_experiment_by_name(config["mlflow"]["experiment_name"])
if experiment:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    print(runs[["run_id", "tags.mlflow.runName", "metrics.best_cv_rmse", "metrics.cv_r2"]].to_string())